# Clasificación multiclase (bcc / mel / scc) con desbalanceo de clases

Lee los CSV de features (`features_multiclass/features_train.csv` y `features_val.csv`) y entrena varios modelos
**teniendo en cuenta el desbalanceo**.

**Para ir rápido al principio** se trabaja con una submuestra estratificada (`N_TRAIN = 2000`, `N_VAL = 1000`)
que mantiene la proporción real de cada clase en ambos splits. Con `N_TRAIN = None` / `N_VAL = None` se usa todo.

## Qué métricas mirar (y cuáles no)
- **Accuracy: no sirve** como métrica principal. Un modelo que siempre dice la clase mayoritaria ya tiene una accuracy alta.
- **ROC-AUC: no está "mal", pero engaña con desbalanceo.** La tasa de falsos positivos se divide entre muchísimos negativos, así que
  casi cualquier modelo sale con un AUC alto. Se muestra solo como dato informativo.
- **PR-AUC (average precision)** es la métrica principal. Se calcula **una-contra-resto por clase** y se hace la **media macro**
  (cada clase pesa lo mismo, aunque sea minoritaria). Su línea base no es 0.5 sino la **prevalencia** de la clase, por eso se da también
  **PR-AUC normalizada** = (AP - prevalencia) / (1 - prevalencia): 0 = azar, 1 = perfecto.
- Para decisiones duras (una clase por imagen): **F1 macro**, **balanced accuracy** (= recall macro), **MCC** y el **recall por clase**
  (en `mel` el recall es lo que más importa clínicamente: un melanoma que se escapa es el peor error).

## Cómo se trata el desbalanceo
1. `class_weight="balanced"` (o pesos por muestra en el boosting): los errores en clases pequeñas cuestan más.
2. **Validación cruzada estratificada**: cada fold conserva la proporción de clases.
3. **Ajuste de pesos por clase sobre las probabilidades OOF** (solo con train): se buscan multiplicadores por clase que maximizan el F1 macro
   y luego se aplican a val. Es el equivalente multiclase de "mover el umbral".
4. **Ensemble** por media de probabilidades de los modelos.

Selección de modelo: por **CV en train** (PR-AUC macro). Val se usa solo para informar, no para elegir.

In [1]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_recall_curve, precision_recall_fscore_support,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, label_binarize
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

# ---------------- Configuración ----------------
ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")
FEAT_DIR = ROOT / "features_multiclass"          # salida de feature_extraction_multiclass
OUT_DIR = ROOT / "results_multiclass"            # resultados de este notebook
CLASSES = ["bcc", "mel", "scc"]                  # el orden define el índice de cada clase (0, 1, 2)
CLASS_TO_LABEL = {c: i for i, c in enumerate(CLASSES)}

N_TRAIN = None           # nº de imágenes de train (estratificado por clase). None = todas
N_VAL = None             # nº de imágenes de val   (estratificado por clase). None = todas
SEED = 42
CV_FOLDS = 5
N_JOBS = 4               # hilos del Random Forest (-1 = todos, -2 = todos menos uno)

MODELS_TO_RUN = ["logreg", "svm", "rf", "hgb"]   # ver make_model()
USE_ENSEMBLE = True      # media de probabilidades de los modelos de arriba
TUNE_CLASS_WEIGHTS = True    # ajusta multiplicadores por clase sobre las probabilidades OOF (maximiza F1 macro)
RUN_ABLATION = False      # ablación por familias de features
ABLATION_MODEL = "rf"

USE_FAMILIES = ["shape", "color", "tex", "gcolor", "gtex"]   # familias de features a usar
PREFIX = {"shape": "shape_", "color": "color_", "tex": "tex_", "gcolor": "gcolor_", "gtex": "gtex_"}

## Carga y submuestra estratificada (misma proporción de clases que el dataset completo)

In [2]:
def allocate(counts, n):
    """Reparto proporcional (metodo del mayor resto) que suma exactamente n."""
    total = sum(counts.values())
    exact = {k: n * v / total for k, v in counts.items()}
    alloc = {k: int(np.floor(v)) for k, v in exact.items()}
    rest = n - sum(alloc.values())
    for k in sorted(exact, key=lambda k: exact[k] - alloc[k], reverse=True)[:rest]:
        alloc[k] += 1
    return alloc


def stratified_subsample(df, n, seed):
    """Submuestra de n filas conservando la proporción de cada clase."""
    if n is None or n >= len(df):
        return df.copy()
    alloc = allocate(df["class"].value_counts().to_dict(), n)
    parts = [g.sample(n=alloc[c], random_state=seed) for c, g in df.groupby("class")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


def class_table(full, sample, title):
    a = full["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
    b = sample["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
    t = pd.DataFrame({"completo": a, "% completo": (100 * a / a.sum()).round(2),
                      "muestra": b, "% muestra": (100 * b / b.sum()).round(2)})
    print(title)
    print(t.to_string(), "\n")


tr_full = pd.read_csv(FEAT_DIR / "features_train.csv")
va_full = pd.read_csv(FEAT_DIR / "features_val.csv")
unknown = (set(tr_full["class"]) | set(va_full["class"])) - set(CLASSES)
assert not unknown, f"Clases en el CSV que no están en CLASSES: {unknown}"

tr = stratified_subsample(tr_full, N_TRAIN, SEED)
va = stratified_subsample(va_full, N_VAL, SEED)
class_table(tr_full, tr, "TRAIN")
class_table(va_full, va, "VAL")
if tr["class"].value_counts().min() < 3 * CV_FOLDS:
    print("[aviso] alguna clase tiene muy pocas imágenes en train para una CV estable: sube N_TRAIN o baja CV_FOLDS")

TRAIN
       completo  % completo  muestra  % muestra
class                                          
bcc        1993       39.22      235      39.17
mel        2713       53.38      320      53.33
scc         376        7.40       45       7.50 

VAL
       completo  % completo  muestra  % muestra
class                                          
bcc         498       39.21      118      39.33
mel         678       53.39      160      53.33
scc          94        7.40       22       7.33 



## Features (solo columnas `shape_`, `color_`, `tex_`, `gcolor_`, `gtex_`; `qc_*`, `name`, `subtype`... no se usan)

In [3]:
def family_of(col):
    for fam, p in PREFIX.items():
        if col.startswith(p):
            return fam
    return None


feat_all = [c for c in tr.columns if family_of(c)]
clean = lambda d: d[feat_all].replace([np.inf, -np.inf], np.nan)
Xtr_raw, Xva_raw = clean(tr), clean(va)

# fuera: columnas casi vacías o constantes (calculado solo con train)
dropped = [c for c in feat_all if Xtr_raw[c].notna().mean() < 0.5 or Xtr_raw[c].nunique() < 2]
feat = [c for c in feat_all if c not in dropped and family_of(c) in USE_FAMILIES]
FAMILY = pd.Series({c: family_of(c) for c in feat})

X_tr, X_va = Xtr_raw[feat], Xva_raw[feat]
y_tr = tr["class"].map(CLASS_TO_LABEL).to_numpy()
y_va = va["class"].map(CLASS_TO_LABEL).to_numpy()
K = len(CLASSES)

print(f"{len(feat)} features usadas ({len(dropped)} descartadas por vacías/constantes)")
print(FAMILY.value_counts().to_string())
print(f"NaN medio: {100 * X_tr.isna().mean().mean():.2f}%")

420 features usadas (0 descartadas por vacías/constantes)
tex       95
gtex      95
color     83
shape     76
gcolor    71
NaN medio: 0.00%


## Métricas y ajuste de pesos por clase

In [4]:
def evaluate(y, P, w=None):
    """Métricas multiclase. P: probabilidades (n, K). w: multiplicadores por clase (None = argmax normal)."""
    pred = (P * (np.ones(K) if w is None else w)).argmax(1)
    Y = label_binarize(y, classes=np.arange(K))
    prev = Y.mean(0)
    ap = np.array([average_precision_score(Y[:, k], P[:, k]) for k in range(K)])
    rec = recall_score(y, pred, average=None, labels=np.arange(K), zero_division=0)
    out = {"PR-AUC macro": ap.mean(),                                  # MÉTRICA PRINCIPAL
           "PR-AUC norm": np.mean((ap - prev) / (1 - prev)),           # 0 = azar, 1 = perfecto
           "F1 macro": f1_score(y, pred, average="macro", zero_division=0),
           "Bal. acc": balanced_accuracy_score(y, pred),
           "MCC": matthews_corrcoef(y, pred),
           "ROC-AUC macro (info)": roc_auc_score(Y, P, average="macro"),
           "Accuracy (info)": accuracy_score(y, pred)}
    for k, c in enumerate(CLASSES):
        out[f"AP {c}"] = ap[k]
        out[f"recall {c}"] = rec[k]
    return out


def tune_class_weights(P, y, grid=np.exp(np.linspace(np.log(0.1), np.log(10), 21)), rounds=3):
    """Busca multiplicadores por clase (la clase 0 es la referencia) que maximizan el F1 macro de argmax(P * w)."""
    f = lambda w: f1_score(y, (P * w).argmax(1), average="macro", zero_division=0)
    w = np.ones(K)
    best = f(w)
    for _ in range(rounds):
        for k in range(1, K):
            for g in grid:
                w2 = w.copy()
                w2[k] = g
                s = f(w2)
                if s > best + 1e-9:
                    best, w = s, w2
    return w / w.sum() * K

## Modelos

In [ ]:
def make_model(name):
    if name == "logreg":    # lineal; QuantileTransformer = robusto a colas pesadas / outliers
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc", QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)),
                         ("clf", LogisticRegression(C=0.05, class_weight="balanced", max_iter=3000))])
    if name == "svm":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc", QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)),
                         ("clf", SVC(C=1.0, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED))])
    if name == "rf":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("clf", RandomForestClassifier(n_estimators=400, min_samples_leaf=2, max_features="sqrt",
                                                        class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=SEED))])
    if name == "hgb":       # gestiona NaN solo; el desbalanceo se trata con pesos por muestra (ver fit_model)
        return Pipeline([("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                                                min_samples_leaf=20, l2_regularization=1.0,
                                                                early_stopping=True, validation_fraction=0.15,
                                                                random_state=SEED))])
    raise ValueError(name)


def fit_model(name, X, y):
    m = make_model(name)
    if name == "hgb":
        m.fit(X, y, clf__sample_weight=compute_sample_weight("balanced", y))
    else:
        m.fit(X, y)
    return m


def oof_proba(name, X, y):
    """Probabilidades out-of-fold con CV estratificada (cada muestra se predice con un modelo que no la ha visto)."""
    skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)

    P = np.zeros((len(y), K))
    for a, b in skf.split(X, y):       # folds en serie (evita los procesos de joblib, que fallan en algunos Windows)
        P[b] = fit_model(name, X.iloc[a], y[a]).predict_proba(X.iloc[b])
    return P


P_oof, P_va, fitted = {}, {}, {}
for name in MODELS_TO_RUN:
    t0 = time.time()
    P_oof[name] = oof_proba(name, X_tr, y_tr)
    fitted[name] = fit_model(name, X_tr, y_tr)
    P_va[name] = fitted[name].predict_proba(X_va)
    print(f"{name:7s} listo en {time.time() - t0:.0f}s")

members = list(MODELS_TO_RUN)
if USE_ENSEMBLE and len(members) > 1:
    P_oof["ensemble"] = np.mean([P_oof[m] for m in members], axis=0)
    P_va["ensemble"] = np.mean([P_va[m] for m in members], axis=0)

# línea base: predecir siempre las prevalencias de train (modelo sin información)
prior = np.bincount(y_tr, minlength=K) / len(y_tr)
P_oof["baseline"] = np.tile(prior, (len(y_tr), 1))
P_va["baseline"] = np.tile(prior, (len(y_va), 1))

## Resultados: CV en train (selección) y val (informe)

In [ ]:
W = {n: (tune_class_weights(P_oof[n], y_tr) if TUNE_CLASS_WEIGHTS and n != "baseline" else np.ones(K)) for n in P_oof}

rows = {}
for n in P_oof:
    cv, v0, v1 = evaluate(y_tr, P_oof[n]), evaluate(y_va, P_va[n]), evaluate(y_va, P_va[n], W[n])
    rows[n] = {"CV PR-AUC": cv["PR-AUC macro"], "CV F1 macro": cv["F1 macro"],
               "VAL PR-AUC": v0["PR-AUC macro"], "VAL PR-AUC norm": v0["PR-AUC norm"],
               "VAL F1 macro": v0["F1 macro"], "VAL F1 macro (pesos aj.)": v1["F1 macro"],
               "VAL Bal.acc (aj.)": v1["Bal. acc"], "VAL MCC (aj.)": v1["MCC"],
               "VAL ROC-AUC (info)": v0["ROC-AUC macro (info)"], "VAL Accuracy (info)": v0["Accuracy (info)"]}
summary = pd.DataFrame(rows).T.sort_values("CV PR-AUC", ascending=False)

best = next(n for n in summary.index if n != "baseline")      # mejor modelo según CV (no según val)
print(f"Mejor modelo por CV PR-AUC macro: {best}\n")
display(summary.round(3))
print("Pesos por clase ajustados (", CLASSES, "):")
for n in summary.index:
    print(f"  {n:9s}", np.round(W[n], 2))

In [ ]:
# Detalle por clase del mejor modelo en val (con los pesos ajustados)
P, w = P_va[best], W[best]
pred = (P * w).argmax(1)
Y = label_binarize(y_va, classes=np.arange(K))
prec, rec, f1, sup = precision_recall_fscore_support(y_va, pred, labels=np.arange(K), zero_division=0)
per_class = pd.DataFrame({"soporte": sup, "prevalencia": Y.mean(0),
                          "AP (PR-AUC)": [average_precision_score(Y[:, k], P[:, k]) for k in range(K)],
                          "precision": prec, "recall": rec, "F1": f1}, index=CLASSES)
per_class["AP / prevalencia"] = per_class["AP (PR-AUC)"] / per_class["prevalencia"]
print(f"Modelo: {best} | val ({len(y_va)} imágenes)")
display(per_class.round(3))


def plot_pr_curves(P, y, title):
    Y = label_binarize(y, classes=np.arange(K))
    fig, ax = plt.subplots(figsize=(6, 5))
    for k, c in enumerate(CLASSES):
        p, r, _ = precision_recall_curve(Y[:, k], P[:, k])
        line, = ax.plot(r, p, label=f"{c}  AP={average_precision_score(Y[:, k], P[:, k]):.2f}  (azar={Y[:, k].mean():.2f})")
        ax.axhline(Y[:, k].mean(), ls="--", lw=0.8, color=line.get_color())
    ax.set(xlabel="Recall", ylabel="Precision", title=title, xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(loc="lower left", fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_confusions(y, pred, title):
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    cmn = cm / cm.sum(1, keepdims=True).clip(1)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, M, fmt, t in [(axs[0], cm, "d", "Conteos"), (axs[1], cmn, ".2f", "Normalizada por fila (= recall)")]:
        ax.imshow(M, cmap="Blues", vmin=0, vmax=None if fmt == "d" else 1)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, format(M[i, j], fmt), ha="center", va="center",
                        color="white" if M[i, j] > M.max() * 0.6 else "black")
        ax.set(xticks=range(K), yticks=range(K), xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel="Predicha", ylabel="Real", title=t)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_pr_curves(P_va[best], y_va, f"Curvas PR en val · {best} (líneas discontinuas = prevalencia)")
plot_confusions(y_va, pred, f"Matriz de confusión en val · {best} (pesos ajustados)")

## Importancia de features (Random Forest) y ablación por familias

In [ ]:
rf = fitted.get("rf") or fit_model("rf", X_tr, y_tr)
imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=feat).sort_values(ascending=False)

colors = {"shape": "tab:blue", "color": "tab:orange", "tex": "tab:green", "gcolor": "tab:red", "gtex": "tab:purple"}
top = imp.head(25)[::-1]
fig, axs = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [3, 1]})
axs[0].barh(top.index, top.values, color=[colors[FAMILY[c]] for c in top.index])
axs[0].set(title="Top 25 features (importancia RF)")
fam_imp = imp.groupby(FAMILY).sum().sort_values()
axs[1].barh(fam_imp.index, fam_imp.values, color=[colors[f] for f in fam_imp.index])
axs[1].set(title="Importancia total por familia")
plt.tight_layout()
plt.show()

In [ ]:
if RUN_ABLATION:
    fams = [f for f in USE_FAMILIES if (FAMILY == f).any()]
    sets = {"todas": feat}
    for f in fams:
        sets[f"solo {f}"] = list(FAMILY.index[FAMILY == f])
    for f in fams:
        sets[f"sin {f}"] = list(FAMILY.index[FAMILY != f])
    sets["solo lesión (shape+color+tex)"] = list(FAMILY.index[FAMILY.isin(["shape", "color", "tex"])])
    sets["solo imagen entera (gcolor+gtex)"] = list(FAMILY.index[FAMILY.isin(["gcolor", "gtex"])])

    rows = {}
    for nm, cols in sets.items():
        if not cols:
            continue
        m = evaluate(y_tr, oof_proba(ABLATION_MODEL, X_tr[cols], y_tr))
        rows[nm] = {"nº features": len(cols), "CV PR-AUC": m["PR-AUC macro"], "CV PR-AUC norm": m["PR-AUC norm"],
                    "CV F1 macro": m["F1 macro"], "CV Bal.acc": m["Bal. acc"], f"AP {CLASSES[1]}": m[f"AP {CLASSES[1]}"]}
    ablation = pd.DataFrame(rows).T.sort_values("CV PR-AUC", ascending=False)
    print(f"Ablación con {ABLATION_MODEL} (CV en train, {len(y_tr)} imágenes)")
    display(ablation.round(3))

## Guardado de resultados

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.round(4).to_csv(OUT_DIR / "summary_models.csv")
per_class.round(4).to_csv(OUT_DIR / "per_class_best_val.csv")
if RUN_ABLATION:
    ablation.round(4).to_csv(OUT_DIR / "ablation_families.csv")
imp.to_csv(OUT_DIR / "feature_importance_rf.csv", header=["importance"])

pred_df = pd.DataFrame(P_va[best], columns=[f"p_{c}" for c in CLASSES])
pred_df.insert(0, "name", va["name"].values)
pred_df.insert(1, "y_true", va["class"].values)
pred_df.insert(2, "y_pred", [CLASSES[i] for i in (P_va[best] * W[best]).argmax(1)])
pred_df.to_csv(OUT_DIR / f"val_predictions_{best}.csv", index=False)

used = members if best == "ensemble" else [best]
joblib.dump({"models": {m: fitted[m] for m in used}, "class_weights": W[best], "features": feat,
             "classes": CLASSES}, OUT_DIR / "best_model.joblib")
print("Guardado en", OUT_DIR)